# ELG Example Notebook

이 노트북은 `elg` 패키지의 공개 API를 **설명 → 코드** 순서로 하나씩 보여준다.
각 셀은 독립적으로 읽기 쉽게 구성했고, 구조체 / 렌더링 / mutation / codec / normalize / metrics / sampler 순서로 정리했다.

## 1. 전체 공개 API import

먼저 이후 예시들에서 사용할 `elg` 공개 API를 한 번에 import한다.

In [9]:
from elg import (
    AtomicNode,
    AtomicSource,
    AtomicType,
    Hypothesis,
    LogicalNode,
    LogicalOp,
    MutationSample,
    Path,
    RelationNode,
    RelationType,
    count_atomics,
    count_logicals,
    count_nodes,
    count_relations,
    fingerprint,
    generate_mutation_candidates,
    get_node_at_path,
    hypothesis_from_dict,
    hypothesis_from_json,
    hypothesis_to_json,
    iter_paths,
    mutate_append_child,
    mutate_logical_operator,
    mutate_relation_type,
    mutate_remove_child,
    mutate_replace_child,
    mutate_replace_subtree,
    mutate_unwrap_not,
    mutate_wrap_not,
    node_from_dict,
    node_to_dict,
    normalize_hypothesis,
    normalize_node,
    render_pretty,
    render_tree,
    replace_at_path,
    sample_mutation,
    tree_depth,
)
import random

## 2. Enum 타입들

`AtomicType`, `AtomicSource`, `LogicalOp`, `RelationType`은 ELG의 닫힌 vocabulary를 나타낸다.

In [ ]:
print(list(AtomicType))
print(list(AtomicSource))
print(list(LogicalOp))
print(list(RelationType))

## 3. AtomicNode 생성

`AtomicNode`는 더 이상 ELG 내부에서 분해하지 않는 opaque leaf proposition이다.

In [ ]:
primitive_atomic = AtomicNode("FUNDING_FEE > 0", type=AtomicType.BOOLEAN, source=AtomicSource.PRIMITIVE)
semantic_atomic = AtomicNode("시장 상태가 불안정하다", type=AtomicType.ABSTRACT, source=AtomicSource.SEMANTIC)

print(primitive_atomic)
print(semantic_atomic)
print(primitive_atomic.kind)

## 4. LogicalNode 생성

`LogicalNode`는 `AND`, `OR`, `NOT` 같은 논리 연산자를 가진다.

In [ ]:
and_node = LogicalNode(LogicalOp.AND, [
    AtomicNode("FUNDING_FEE > 0"),
    AtomicNode("CLOSE > SMA_20"),
])
not_node = LogicalNode(LogicalOp.NOT, [AtomicNode("RETURN_5M > 0")])

print(and_node)
print(not_node)
print(and_node.kind, and_node.op)

## 5. RelationNode 생성

`RelationNode`는 조건과 결과를 연결하는 관계 노드다.

In [ ]:
relation = RelationNode(
    RelationType.IMPLIES,
    [and_node, AtomicNode("RETURN_5M > 0")],
)

print(relation)
print(relation.kind)
print("condition =", relation.condition)
print("target =", relation.target)

## 6. Hypothesis 생성

`Hypothesis`는 root node 하나를 감싸는 최상위 wrapper다.

In [ ]:
hypothesis = Hypothesis(root=relation, params={"name": "funding-trend hypothesis"})
print(hypothesis)

## 7. node_to_dict

`node_to_dict`는 단일 node를 dict로 직렬화한다.

In [ ]:
print(node_to_dict(and_node))
print(node_to_dict(relation))

## 8. hypothesis_to_json

`hypothesis_to_json`은 hypothesis 전체를 JSON 문자열로 만든다.

In [ ]:
payload_json = hypothesis_to_json(hypothesis)
print(payload_json)

## 9. node_from_dict

`node_from_dict`는 `kind`를 기준으로 atomic / logical / relation node를 복원한다.

In [ ]:
node_payload = {
    "kind": "logical",
    "op": "OR",
    "inputs": [
        {"kind": "atomic", "name": "A", "type": "abstract", "source": "semantic", "params": {}},
        {"kind": "atomic", "name": "B", "type": "abstract", "source": "semantic", "params": {}},
    ],
    "params": {},
}
restored_node = node_from_dict(node_payload)
print(restored_node)
print(render_pretty(restored_node))

## 10. hypothesis_from_dict

dict payload를 다시 `Hypothesis` 객체로 복원할 수 있다.

In [ ]:
hypothesis_payload = hypothesis.to_dict()
restored_hypothesis = hypothesis_from_dict(hypothesis_payload)
print(restored_hypothesis)
print(render_pretty(restored_hypothesis))

## 11. hypothesis_from_json

JSON 문자열로부터 hypothesis를 복원한다.

In [ ]:
restored_from_json = hypothesis_from_json(payload_json)
print(restored_from_json)
print(render_pretty(restored_from_json))

## 12. render_pretty

`render_pretty`는 사람이 읽기 좋은 중첩 표현으로 ELG를 보여준다.

In [ ]:
print(render_pretty(hypothesis))

## 13. render_tree

`render_tree`는 ASCII tree 형태로 ELG 구조를 시각화한다.

In [ ]:
print(render_tree(hypothesis))

## 14. normalize_node

단일 node를 정규화한다. 예를 들어 중복 child 제거, child 정렬, double negation 제거 같은 처리를 한다.

In [ ]:
unnormalized_node = LogicalNode("AND", [
    AtomicNode("B"),
    AtomicNode("A"),
    AtomicNode("A"),
])
normalized_node = normalize_node(unnormalized_node)
print(render_pretty(unnormalized_node))
print('---')
print(render_pretty(normalized_node))

## 15. normalize_hypothesis

전체 hypothesis를 정규화한다.

In [ ]:
unnormalized_hypothesis = Hypothesis(
    root=LogicalNode("NOT", [LogicalNode("NOT", [AtomicNode("A")])])
)
normalized_hypothesis = normalize_hypothesis(unnormalized_hypothesis)
print(render_pretty(unnormalized_hypothesis))
print('---')
print(render_pretty(normalized_hypothesis))

## 16. 구조 메트릭: count_nodes, tree_depth, count_atomics, count_logicals, count_relations

ELG의 구조적 복잡도를 정량화하는 기본 함수들이다.

In [ ]:
print("count_nodes =", count_nodes(hypothesis))
print("tree_depth =", tree_depth(hypothesis))
print("count_atomics =", count_atomics(hypothesis))
print("count_logicals =", count_logicals(hypothesis))
print("count_relations =", count_relations(hypothesis))

## 17. fingerprint

`fingerprint`는 정규화된 구조를 기반으로 해시를 만든다. 구조적으로 같은 가설은 같은 fingerprint를 갖는다.

In [ ]:
h1 = Hypothesis(root=LogicalNode("AND", [AtomicNode("A"), AtomicNode("B")]))
h2 = Hypothesis(root=LogicalNode("AND", [AtomicNode("B"), AtomicNode("A")]))

print(fingerprint(h1))
print(fingerprint(h2))
print("same fingerprint:", fingerprint(h1) == fingerprint(h2))

## 18. Path 타입

`Path`는 트리 안의 특정 위치를 가리키는 tuple 기반 주소다.

- `()` = root
- `(0,)` = 첫 번째 child
- `(0, 1)` = root의 첫 child의 두 번째 child

In [ ]:
root_path: Path = ()
condition_path: Path = (0,)
second_condition_child_path: Path = (0, 1)

a = root_path, condition_path, second_condition_child_path
print(a)

## 19. get_node_at_path

주어진 path의 node를 가져온다.

In [ ]:
print(get_node_at_path(hypothesis, ()))
print(get_node_at_path(hypothesis, (0,)))
print(get_node_at_path(hypothesis, (0, 1)))

## 20. iter_paths

현재 hypothesis에서 접근 가능한 모든 path를 순회한다.

In [ ]:
print(iter_paths(hypothesis))

## 21. replace_at_path

지정한 path의 subtree를 새 node로 교체한다. 원본 hypothesis는 바뀌지 않는다.

In [ ]:
replaced = replace_at_path(hypothesis, (0, 1), AtomicNode("OPEN_INTEREST_CHANGE > 0"))
print(render_pretty(replaced))
print('--- original ---')
print(render_pretty(hypothesis))

## 22. mutate_replace_subtree

`replace_at_path`의 mutation-friendly wrapper로, subtree 전체를 새 구조로 바꾼다.

In [ ]:
subtree_mutated = mutate_replace_subtree(
    hypothesis,
    (0,),
    LogicalNode("OR", [AtomicNode("VOLATILITY_HIGH"), AtomicNode("FUNDING_FEE > 0")]),
)
print(render_pretty(subtree_mutated))

## 23. mutate_replace_child

논리 노드나 관계 노드의 특정 child만 교체한다.

In [ ]:
child_mutated = mutate_replace_child(hypothesis, (0,), 1, AtomicNode("VOLUME > AVG_VOLUME"))
print(render_pretty(child_mutated))

## 24. mutate_logical_operator

`AND -> OR`, `OR -> AND` 같은 logical operator 변경을 수행한다.

In [ ]:
logical_mutated = mutate_logical_operator(hypothesis, (0,), "OR")
print(render_pretty(logical_mutated))

## 25. mutate_relation_type

relation 종류를 바꾼다.

In [ ]:
relation_mutated = mutate_relation_type(hypothesis, (), "SUPPORT")
print(render_pretty(relation_mutated))

## 26. mutate_wrap_not

지정한 path의 node를 `NOT(node)`로 감싼다.

In [ ]:
wrapped_not = mutate_wrap_not(hypothesis, (1,))
print(render_pretty(wrapped_not))

## 27. mutate_unwrap_not

`NOT(A)`를 다시 `A`로 푼다.

In [ ]:
unwrapped = mutate_unwrap_not(wrapped_not, (1,))
print(render_pretty(unwrapped))

## 28. mutate_append_child

`AND/OR` 노드에 child를 하나 더 추가한다.

In [ ]:
appended = mutate_append_child(hypothesis, (0,), AtomicNode("VOLUME > AVG_VOLUME"))
print(render_pretty(appended))

## 29. mutate_remove_child

`AND/OR` 노드에서 특정 child를 제거한다. 최소 arity를 깨면 예외가 난다.

In [ ]:
removed = mutate_remove_child(appended, (0,), 1)
print(render_pretty(removed))

## 30. MutationSample

샘플러는 결과를 `MutationSample`로 반환한다. 안에는 어떤 mutation이 선택됐는지 메타데이터가 들어 있다.

In [ ]:
seeded_sample = MutationSample(
    operation="manual_demo",
    path=(0,),
    result=hypothesis,
    details={"note": "demo object"},
)
print(seeded_sample)

## 31. generate_mutation_candidates

현재 hypothesis에서 가능한 legal mutation 후보들을 전부 생성한다.

In [ ]:
candidates = generate_mutation_candidates(hypothesis, atomic_pool=["X", "D"])
print("candidate count =", len(candidates))
for item in candidates[:5]:
    print(item.operation, item.path, item.details)

## 32. sample_mutation

후보들 중 하나를 랜덤하게 샘플링한다. `random.Random(seed)`를 넣으면 재현 가능하다.

In [ ]:
sampled = sample_mutation(hypothesis, rng=random.Random(7), atomic_pool=["X", "D"])
print(sampled.operation)
print(sampled.path)
print(sampled.details)
print(render_pretty(sampled.result))

## 33. 전체 흐름 예시

마지막으로 ELG를 한 번에 생성하고, 렌더링하고, mutate하고, serialize하고, normalize/fingerprint를 계산하는 전체 흐름을 보여준다.

In [ ]:
base = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode("AND", [
                AtomicNode("FUNDING_FEE > 0"),
                AtomicNode("CLOSE > SMA_20"),
            ]),
            AtomicNode("RETURN_5M > 0"),
        ],
    ),
    params={"name": "end-to-end demo"},
)

print('PRETTY')
print(render_pretty(base))
print('---')
print('TREE')
print(render_tree(base))
print('---')
mutated = sample_mutation(base, rng=random.Random(3), atomic_pool=["VOLATILITY_HIGH", "OPEN_INTEREST_CHANGE > 0"])
print('SAMPLED MUTATION =', mutated.operation, mutated.path, mutated.details)
print(render_pretty(mutated.result))
print('---')
json_payload = hypothesis_to_json(mutated.result)
print(json_payload)
print('---')
restored = hypothesis_from_json(json_payload)
print('fingerprint =', fingerprint(restored))
print('normalized =')
print(render_pretty(normalize_hypothesis(restored)))
print('metrics =', count_nodes(restored), tree_depth(restored), count_atomics(restored), count_logicals(restored), count_relations(restored))

## 34. LLMClient with OpenRouter

이 예시는 `hypoevolve.llm.LLMClient`를 **OpenRouter**와 함께 사용하는 방법을 보여준다.

- backend: OpenRouter (OpenAI-compatible)
- model: `deepseek/deepseek-v3.2`

> 보안상 **실제 API 키를 노트북에 직접 저장하지 말고**, 환경변수로 주입하는 방식을 사용한다.

In [10]:
import os
from hypoevolve import LLMClient, LLMConfig

# 권장: 셸에서 먼저 export OPENAI_API_KEY=... 해두기
# 또는 이 셀에서 임시로 설정하기 (노트북 저장 전 제거 권장)
# os.environ["OPENAI_API_KEY"] = "sk-or-..."
API_KEY = "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"

llm_config = LLMConfig(
    model="deepseek/deepseek-v3.2",
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    temperature=0.2,
    max_tokens=1000,
    timeout=60,
    retries=2,
    retry_delay=1.0,
)

client = LLMClient(llm_config)
print(llm_config)

LLMConfig(model='deepseek/deepseek-v3.2', temperature=0.2, max_tokens=1000, api_key='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', api_base='https://openrouter.ai/api/v1', timeout=60, retries=2, retry_delay=1.0)


## 35. generate_text 예시

`generate_text()`는 system prompt와 user prompt를 받아 일반 텍스트 응답을 반환한다.

In [4]:
system_prompt = "You are a concise research assistant."
user_prompt = "Summarize why structured hypothesis mutation can be more stable than raw natural-language mutation in 3 bullets."

# 실제 호출 예시
response = client.generate_text(system_prompt, user_prompt)
print(response)

print("client.generate_text(system_prompt, user_prompt)")

- **Reduced Ambiguity:** Structured formats enforce clear, testable components (e.g., variables, relationships), minimizing the misinterpretation and inconsistency common in free-text natural language.
- **Controlled Variation:** Mutations can be applied systematically to specific elements (like parameters or operators), ensuring logical coherence and preventing nonsensical or irrelevant changes.
- **Automation & Consistency:** Structured hypotheses are machine-readable, enabling reliable automated generation, evaluation, and iteration, which reduces human error and bias over time.
client.generate_text(system_prompt, user_prompt)


## 36. generate_json 예시

`generate_json()`는 LLM에게 JSON만 반환하도록 지시하고, 그 응답을 Python `dict`로 파싱한다.

실전에서는 출력 계약을 분명히 적는 것이 중요하다.

In [6]:
system_prompt = "Return only valid JSON."
user_prompt = """
Return a JSON object with keys:
- score: float between 0 and 1
- rationale: short string
for the hypothesis: 'If funding fee is positive then short-term returns are positive.'
"""

# 실제 호출 예시
payload = client.generate_json(system_prompt, user_prompt)
print(payload)

print("client.generate_json(system_prompt, user_prompt)")

{'score': 0.3, 'rationale': 'Funding fee direction does not guarantee short-term return direction'}
client.generate_json(system_prompt, user_prompt)


In [7]:
payload

{'score': 0.3,
 'rationale': 'Funding fee direction does not guarantee short-term return direction'}

## 40. 실제 LLM parser 호출 예시

이 셀은 `LLMClient`와 `llm_parse_hypothesis()`를 실제로 연결해서, 자연어 가설을 ELG로 변환하는 예시다.

- backend: OpenRouter
- model: `deepseek/deepseek-v3.2`

> 실행 전에 `OPENAI_API_KEY` 환경변수가 설정되어 있어야 한다.

In [1]:
from hypoevolve import LLMClient, LLMConfig, llm_parse_hypothesis
from elg import render_pretty, render_tree

API_KEY = "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"

llm_config = LLMConfig(
    model="deepseek/deepseek-v3.2",
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,  # 환경변수 OPENAI_API_KEY 사용
    temperature=0.2,
    max_tokens=1200,
    timeout=60,
    retries=2,
    retry_delay=1.0,
)

client = LLMClient(llm_config)

## 41. 자연어 가설을 ELG로 파싱해보기

아래 셀은 실제로 자연어 가설 문자열을 LLM parser로 보내고, 반환된 ELG를 pretty/tree 형태로 확인하는 예시다.

In [2]:
hypothesis_text = "If funding fee is positive and price is above SMA20, then short-term returns are positive."

# 실제 호출
parsed_hypothesis = llm_parse_hypothesis(hypothesis_text, llm=client, retries=2)

print("=== PRETTY ===")
print(render_pretty(parsed_hypothesis))
print()
print("=== TREE ===")
print(render_tree(parsed_hypothesis))
print()
print("=== RAW DICT ===")
print(parsed_hypothesis.to_dict())

=== PRETTY ===
IMPLIES(
  AND(
    funding fee is positive,
    price is above SMA20
  ),
  short-term returns are positive
)

=== TREE ===
Hypothesis
└── IMPLIES
    ├── AND
    │   ├── funding fee is positive
    │   └── price is above SMA20
    └── short-term returns are positive

=== RAW DICT ===
{'root': {'kind': 'relation', 'type': 'IMPLIES', 'inputs': [{'kind': 'logical', 'op': 'AND', 'inputs': [{'kind': 'atomic', 'name': 'funding fee is positive', 'type': 'abstract', 'source': 'semantic', 'params': {}}, {'kind': 'atomic', 'name': 'price is above SMA20', 'type': 'abstract', 'source': 'semantic', 'params': {}}], 'params': {}}, {'kind': 'atomic', 'name': 'short-term returns are positive', 'type': 'abstract', 'source': 'semantic', 'params': {}}], 'params': {}}, 'params': {}}


In [6]:

try:
    parsed_hypothesis = llm_parse_hypothesis(hypothesis_text, llm=client, retries=2)
except Exception as e:
    print(type(e).__name__, e)
    if hasattr(e, "errors"):
        print("---- detailed errors ----")
        for item in e.errors:
            print(item)

ParseError Failed to convert natural-language hypothesis to ELG via LLM
---- detailed errors ----
attempt 1: Unsupported logical operator: None
attempt 2: Unsupported logical operator: None
attempt 3: Unsupported logical operator: None


In [7]:
from hypoevolve.parser import PARSER_SYSTEM_PROMPT, PARSER_RETRY_PROMPT

system_prompt = PARSER_SYSTEM_PROMPT
user_prompt = f"Convert this natural-language hypothesis into ELG JSON root node:\n\n{hypothesis_text}"

raw = client.generate_text(system_prompt, user_prompt)
print(raw)

payload = client.generate_json(system_prompt, user_prompt)
print(payload)

```json
{
  "kind": "relation",
  "type": "IMPLIES",
  "inputs": [
    {
      "kind": "logical",
      "operator": "AND",
      "inputs": [
        {
          "kind": "atomic",
          "params": {}
        },
        {
          "kind": "atomic",
          "params": {}
        }
      ]
    },
    {
      "kind": "atomic",
      "params": {}
    }
  ]
}
```
{'kind': 'relation', 'type': 'IMPLIES', 'inputs': [{'kind': 'logical', 'operator': 'AND', 'inputs': [{'kind': 'atomic', 'proposition': 'funding fee is positive', 'params': {}}, {'kind': 'atomic', 'proposition': 'price is above SMA20', 'params': {}}], 'params': {}}, {'kind': 'atomic', 'proposition': 'short-term returns are positive', 'params': {}}], 'params': {}}
